# Client demo

Demonstrate the use of the Client (Assetto Corsa Sim interface) without the OpenAI Gym interface layer.

In [2]:
%matplotlib inline

import matplotlib.pyplot as plt
import sys
import os
import numpy as np
import pandas as pd
import glob as glob
import time
import pickle
from omegaconf import OmegaConf

# add custom paths
sys.path.extend([os.path.abspath('./assetto_corsa_gym')])
import AssettoCorsaEnv.assettoCorsa as assettoCorsa

# Configure the logging system
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,  # Set the logging level (DEBUG, INFO, WARNING, ERROR, CRITICAL)
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',  # Format of the log messages
    datefmt='%Y-%m-%d %H:%M:%S',  # Format of the timestamp
)


# load config file

In [4]:
config = OmegaConf.load("config.yml")
client = assettoCorsa.make_client_only(config.AssettoCorsa)

# Show static info

In [5]:
static_info = client.simulation_management.get_static_info()
ac_mod_config = client.simulation_management.get_config()

logger.info("Static info:")
for i in static_info:
    logger.info(f"{i}: {static_info[i]}")
logger.info("AC Mod config:")
for i in ac_mod_config:
    logger.info(f"{i}: {ac_mod_config[i]}")


INFO:__main__:Static info:
INFO:__main__:TyreContactPoint_RL: [-208.6613006591797, -4.52930212020874, 689.62744140625]
INFO:__main__:isCarInPit: 0
INFO:__main__:TrackConfiguration: gp
INFO:__main__:CarName: bmw_z4_gt3
INFO:__main__:CAR_WHEELBASE: 2.510283067639306
INFO:__main__:TyreContactPoint_FR: [-208.70315551757812, -4.5960893630981445, 686.6282348632812]
INFO:__main__:TrackName: ks_silverstone
INFO:__main__:CAR_WHEEL_R: [0.3400000035762787, 0.3400000035762787, 0.35249999165534973, 0.35249999165534973]
INFO:__main__:CAR_TRACK: 1.6656040983126834
INFO:__main__:TrackLength: 5803.99658203125
INFO:__main__:autoShifterOn: 1
INFO:__main__:WindSpeed: 0
INFO:__main__:WorldPosition: [-208.71771240234375, -4.1741719245910645, 688.0503540039062]
INFO:__main__:LastSplits: [0, 0, 0]
INFO:__main__:WindDirection: 359
INFO:__main__:TyreContactPoint_FL: [-210.04676818847656, -4.5846781730651855, 687.5348510742188]
INFO:__main__:penaltiesEnabled: 1
INFO:__main__:TyreContactPoint_RR: [-207.2806701660

# Test gear shift

In [6]:
# test gear shifting
client.controls.set_controls(0,-1,1, enable_gear_shift=True, shift_up=True)
client.controls.apply_local_controls()
time.sleep(0.01)
client.controls.set_controls(0,-1,1, enable_gear_shift=True, shift_up=False)
client.controls.apply_local_controls()

time.sleep(1.0)
client.controls.set_controls(0,-1,1, enable_gear_shift=True, shift_down=True)
client.controls.apply_local_controls()
time.sleep(0.01)
client.controls.set_controls(0,-1,1, enable_gear_shift=True, shift_down=False)
client.controls.apply_local_controls()

# Apply actions and recover the car

In [7]:
states = []

client.reset()

# shift up
client.controls.set_controls(0,-1,1, enable_gear_shift=True, shift_up=True)
client.controls.apply_local_controls()
time.sleep(0.01)
client.controls.set_controls(0,-1,1, enable_gear_shift=True, shift_up=False)
client.controls.apply_local_controls()

for i in range(100):
    if i % 2 == 0:
        steer = .1
    else:
        steer = -.1
    client.controls.set_controls(steer=steer, acc=0.5, brake=-1.)
    client.respond_to_server()
    state = client.step_sim()
    states.append(state.copy())
    time.sleep(0.01)

client.controls.set_defaults()
client.respond_to_server()
client.simulation_management.send_reset()
client.close()

INFO:AssettoCorsaEnv.ac_client:sending reset to simulation management server
INFO:AssettoCorsaEnv.ac_client:AC Client. Listening at host: localhost port: 2345
INFO:AssettoCorsaEnv.ac_client:Client connected on 2345
INFO:AssettoCorsaEnv.ac_client:sending reset to simulation management server


# Plot states

In [8]:
df = pd.DataFrame(states)
df.columns

Index(['done', 'tyre_slip_ratio_rr', 'actualGear', 'tyre_slip_ratio_rl',
       'tyreContactHeading_FL_x', 'tyreContactHeading_FL_y',
       'tyreContactHeading_FL_z', 'NdSlip', 'fr_tire_pressure', 'Dy_fl',
       'rl_damper_linear_potentiometer', 'yaw', 'CamberRad', 'iLastTime',
       'world_position_x', 'completedLaps', 'world_position_z', 'timestamp_ac',
       'world_position_y', 'rl_wheel_load', 'Dy_fr', 'LastFF',
       'rl_tire_pressure', 'tyre_loaded_radius', 'tyre_slip_ratio_fr',
       'tyres_load', 'rr_tire_pressure', 'cgHeight', 'accStatus',
       'rr_wheel_load', 'fl_tire_pressure', 'tyreContactHeading_FR_y', 'speed',
       'pitch', 'tyre_slip_ratio_fl', 'packetId', 'isInPit', 'brakeStatus',
       'local_velocity_y', 'local_velocity_x', 'local_velocity_z', 'Dy_rl',
       'Mz', 'LapInvalidated', 'velocity_z', 'Dy_rr', 'velocity_x',
       'fl_damper_linear_potentiometer', 'tyreContactHeading_FR_z',
       'tyreContactHeading_FR_x', 'BestLap', 'RPM', 'fl_tire_temperatur

In [ ]:
plt.title("Speed")
plt.plot(df.speed)
plt.show()

plt.title("steerAngle")
plt.plot(df.steerAngle)
plt.show()